# Formation of UXTD Metadata (Normal Speech)

In [1]:
# importing all libraries
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import soundfile as sf
from IPython.display import Audio, display


In [2]:
# reading the speaker information file and making a dataframe
uxtd_info_df=pd.read_csv("UXTD_info.csv")
uxtd_info_df.head()

,SPEAKER-ID,GROUP,GENDER,AGE-Y,AGE-M,AGE,RECORDING DATE,Unnamed: 7
0,01M,1,M,11,10,11.83,13/11/2011,NaN
1,02M,1,M,11,9,11.75,22/12/2011,NaN
2,03F,1,F,10,5,10.42,27/1/2011,NaN
3,04M,1,M,8,9,8.75,27/1/2011,NaN
4,05M,1,M,9,11,9.92,3/2/2012,NaN


In [3]:
uxtd_info_df.shape

(58, 8)

In [4]:
# checking for any missing values
uxtd_info_df.isna().sum()

SPEAKER-ID         0
GROUP              0
GENDER             0
AGE-Y              0
AGE-M              0
AGE                0
RECORDING DATE     0
Unnamed: 7        58
dtype: int64

In [5]:
uxtd_info_df = uxtd_info_df.drop(columns=["Unnamed: 7"])

In [6]:
uxtd_info_df.isna().sum()

SPEAKER-ID        0
GROUP             0
GENDER            0
AGE-Y             0
AGE-M             0
AGE               0
RECORDING DATE    0
dtype: int64

In [7]:
# now forming a metatable of the data based inside core-uxtd directory

dataset = "uxtd" # storing dataset name as a string
dataset_root = Path("core-uxtd") # storing the folder path name

# sub-folders inside the root folder
core_dir = dataset_root/ "core"
lab_dir = dataset_root/ "speaker_labels"/ "lab"

rows = []

# now looping through dataset

# looping through each speaker folder
for speaker_dir in sorted(core_dir.iterdir()): # iterate is used to make a iterative generator for the for loop because core_dir is just a path
    if not speaker_dir.is_dir(): # check for folder only
        continue # skip if not found

    speaker_id = speaker_dir.name # taking the folder name



    for wav_path in sorted(speaker_dir.glob("*.wav")):
        file_id = wav_path.stem # skip the extension part

        txt_path = speaker_dir/ f"{file_id}.txt"
        full_id = f"{speaker_id}-{file_id}" # pattern to match the text file format
        lab_path = lab_dir/ f"{full_id}.lab"

        rows.append({
                "dataset_name": dataset,
                "speaker_id": speaker_id,
                "session_id": "No Therapy Needed",
                "file_id" : file_id, 
                "full_id": full_id,
                "raw_wav_path": str(wav_path),
                "txt_path": str(txt_path) if txt_path.exists() else None,
                "speaker_label_path": str(lab_path) if lab_path.exists() else None,
                "class_label": 0
            })



uxtd_metadata=pd.DataFrame(rows)
uxtd_metadata.head()


,dataset_name,speaker_id,session_id,file_id,full_id,raw_wav_path,txt_path,speaker_label_path,class_label
0,uxtd,01M,No Therapy Needed,001B,01M-001B,core-uxtd/core/01M/001B.wav,core-uxtd/core/01M/001B.txt,core-uxtd/speaker_labels/lab/01M-001B.lab,0
1,uxtd,01M,No Therapy Needed,002B,01M-002B,core-uxtd/core/01M/002B.wav,core-uxtd/core/01M/002B.txt,core-uxtd/speaker_labels/lab/01M-002B.lab,0
2,uxtd,01M,No Therapy Needed,003B,01M-003B,core-uxtd/core/01M/003B.wav,core-uxtd/core/01M/003B.txt,core-uxtd/speaker_labels/lab/01M-003B.lab,0
3,uxtd,01M,No Therapy Needed,004B,01M-004B,core-uxtd/core/01M/004B.wav,core-uxtd/core/01M/004B.txt,core-uxtd/speaker_labels/lab/01M-004B.lab,0
4,uxtd,01M,No Therapy Needed,005B,01M-005B,core-uxtd/core/01M/005B.wav,core-uxtd/core/01M/005B.txt,core-uxtd/speaker_labels/lab/01M-005B.lab,0


In [8]:
uxtd_metadata.isna().sum()

dataset_name            0
speaker_id              0
session_id              0
file_id                 0
full_id                 0
raw_wav_path            0
txt_path                0
speaker_label_path    326
class_label             0
dtype: int64

In [9]:
# checking for column names
print(uxtd_info_df.columns)
print(uxtd_metadata.columns)

Index(['SPEAKER-ID', 'GROUP', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE',
       'RECORDING DATE'],
      dtype='str')
Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label'],
      dtype='str')


In [10]:
# renaming the speaker_id
uxtd_info_df=uxtd_info_df.rename(columns={"SPEAKER-ID":"speaker_id"})

In [11]:
print(uxtd_info_df.columns)
print(uxtd_metadata.columns)

Index(['speaker_id', 'GROUP', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE',
       'RECORDING DATE'],
      dtype='str')
Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label'],
      dtype='str')


In [12]:
# merging the info data with the metadata

uxtd_metadata=uxtd_metadata.merge(
    uxtd_info_df,
    on = "speaker_id",
    how = "left")

In [13]:
uxtd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,full_id,raw_wav_path,txt_path,speaker_label_path,class_label,GROUP,GENDER,AGE-Y,AGE-M,AGE,RECORDING DATE
0,uxtd,01M,No Therapy Needed,001B,01M-001B,core-uxtd/core/01M/001B.wav,core-uxtd/core/01M/001B.txt,core-uxtd/speaker_labels/lab/01M-001B.lab,0,1,M,11,10,11.83,13/11/2011
1,uxtd,01M,No Therapy Needed,002B,01M-002B,core-uxtd/core/01M/002B.wav,core-uxtd/core/01M/002B.txt,core-uxtd/speaker_labels/lab/01M-002B.lab,0,1,M,11,10,11.83,13/11/2011
2,uxtd,01M,No Therapy Needed,003B,01M-003B,core-uxtd/core/01M/003B.wav,core-uxtd/core/01M/003B.txt,core-uxtd/speaker_labels/lab/01M-003B.lab,0,1,M,11,10,11.83,13/11/2011
3,uxtd,01M,No Therapy Needed,004B,01M-004B,core-uxtd/core/01M/004B.wav,core-uxtd/core/01M/004B.txt,core-uxtd/speaker_labels/lab/01M-004B.lab,0,1,M,11,10,11.83,13/11/2011
4,uxtd,01M,No Therapy Needed,005B,01M-005B,core-uxtd/core/01M/005B.wav,core-uxtd/core/01M/005B.txt,core-uxtd/speaker_labels/lab/01M-005B.lab,0,1,M,11,10,11.83,13/11/2011


In [14]:
uxtd_info_df.columns

Index(['speaker_id', 'GROUP', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE',
       'RECORDING DATE'],
      dtype='str')

In [15]:
uxtd_metadata.columns

Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'full_id',
       'raw_wav_path', 'txt_path', 'speaker_label_path', 'class_label',
       'GROUP', 'GENDER', 'AGE-Y', 'AGE-M', 'AGE', 'RECORDING DATE'],
      dtype='str')

In [16]:
# rearranging

uxtd_metadata = uxtd_metadata[
    [
        "dataset_name",
        "speaker_id",
        "session_id",
        "file_id",
        "GENDER",
        "AGE-Y",
        "AGE-M",
        "AGE",
        "full_id",
        "raw_wav_path",
        "txt_path",
        "speaker_label_path",
        "class_label"
    ]
]
uxtd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,txt_path,speaker_label_path,class_label
0,uxtd,01M,No Therapy Needed,001B,M,11,10,11.83,01M-001B,core-uxtd/core/01M/001B.wav,core-uxtd/core/01M/001B.txt,core-uxtd/speaker_labels/lab/01M-001B.lab,0
1,uxtd,01M,No Therapy Needed,002B,M,11,10,11.83,01M-002B,core-uxtd/core/01M/002B.wav,core-uxtd/core/01M/002B.txt,core-uxtd/speaker_labels/lab/01M-002B.lab,0
2,uxtd,01M,No Therapy Needed,003B,M,11,10,11.83,01M-003B,core-uxtd/core/01M/003B.wav,core-uxtd/core/01M/003B.txt,core-uxtd/speaker_labels/lab/01M-003B.lab,0
3,uxtd,01M,No Therapy Needed,004B,M,11,10,11.83,01M-004B,core-uxtd/core/01M/004B.wav,core-uxtd/core/01M/004B.txt,core-uxtd/speaker_labels/lab/01M-004B.lab,0
4,uxtd,01M,No Therapy Needed,005B,M,11,10,11.83,01M-005B,core-uxtd/core/01M/005B.wav,core-uxtd/core/01M/005B.txt,core-uxtd/speaker_labels/lab/01M-005B.lab,0


In [17]:
uxtd_metadata.isna().sum()

dataset_name            0
speaker_id              0
session_id              0
file_id                 0
GENDER                  0
AGE-Y                   0
AGE-M                   0
AGE                     0
full_id                 0
raw_wav_path            0
txt_path                0
speaker_label_path    326
class_label             0
dtype: int64

## Reading the prompt text for txt file

In [18]:
# reading text from the text files

def read_prompt_text(txt_path):
    if pd.isna(txt_path):
        return None
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.readline().strip()
    except Exception:
        return None

uxtd_metadata["prompt_text"] = uxtd_metadata["txt_path"].apply(read_prompt_text) # function will apply on each value on the column
uxtd_metadata.tail(10)

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,txt_path,speaker_label_path,class_label,prompt_text
4599,uxtd,58F,No Therapy Needed,045D,F,9,1,9.08,58F-045D,core-uxtd/core/58F/045D.wav,core-uxtd/core/58F/045D.txt,core-uxtd/speaker_labels/lab/58F-045D.lab,0,double-articulation post-test aCa
4600,uxtd,58F,No Therapy Needed,046D,F,9,1,9.08,58F-046D,core-uxtd/core/58F/046D.wav,core-uxtd/core/58F/046D.txt,core-uxtd/speaker_labels/lab/58F-046D.lab,0,retroflex-fricative post-test no-context
4601,uxtd,58F,No Therapy Needed,047D,F,9,1,9.08,58F-047D,core-uxtd/core/58F/047D.wav,core-uxtd/core/58F/047D.txt,core-uxtd/speaker_labels/lab/58F-047D.lab,0,retroflex-fricative post-test aCa
4602,uxtd,58F,No Therapy Needed,048D,F,9,1,9.08,58F-048D,core-uxtd/core/58F/048D.wav,core-uxtd/core/58F/048D.txt,core-uxtd/speaker_labels/lab/58F-048D.lab,0,close-back-unrounded-vowel post-test no-context
4603,uxtd,58F,No Therapy Needed,049D,F,9,1,9.08,58F-049D,core-uxtd/core/58F/049D.wav,core-uxtd/core/58F/049D.txt,core-uxtd/speaker_labels/lab/58F-049D.lab,0,close-back-unrounded-vowel post-test dV
4604,uxtd,58F,No Therapy Needed,050D,F,9,1,9.08,58F-050D,core-uxtd/core/58F/050D.wav,core-uxtd/core/58F/050D.txt,core-uxtd/speaker_labels/lab/58F-050D.lab,0,close-back-rounded-vowel post-test no-context
4605,uxtd,58F,No Therapy Needed,051D,F,9,1,9.08,58F-051D,core-uxtd/core/58F/051D.wav,core-uxtd/core/58F/051D.txt,core-uxtd/speaker_labels/lab/58F-051D.lab,0,close-back-rounded-vowel post-test dV
4606,uxtd,58F,No Therapy Needed,052D,F,9,1,9.08,58F-052D,core-uxtd/core/58F/052D.wav,core-uxtd/core/58F/052D.txt,core-uxtd/speaker_labels/lab/58F-052D.lab,0,close-front-round-vowel post-test no-context
4607,uxtd,58F,No Therapy Needed,053D,F,9,1,9.08,58F-053D,core-uxtd/core/58F/053D.wav,core-uxtd/core/58F/053D.txt,core-uxtd/speaker_labels/lab/58F-053D.lab,0,close-front-round-vowel post-test dV
4608,uxtd,58F,No Therapy Needed,054E,F,9,1,9.08,58F-054E,core-uxtd/core/58F/054E.wav,core-uxtd/core/58F/054E.txt,core-uxtd/speaker_labels/lab/58F-054E.lab,0,swallow


## Audio Segmentation using speaker_label_path


In [19]:
def diarize_with_speaker_labels(wav_path, lab_path, output_path,
                                keep_label="CHILD",
                                margin_ms=20,
                                min_seg_ms=100):

    audio, sr = sf.read(wav_path)

    # convert stereo to mono if needed
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    margin_samples = round(sr * margin_ms / 1000)
    min_seg_samples = round(sr * min_seg_ms / 1000)

    kept_segments = []
    segment_info = []
    found_child = False
    found_slt = False

    with open(lab_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 3:
                continue

            start_htk = int(parts[0])
            end_htk = int(parts[1])
            label = parts[2].strip().upper()

            if label == "CHILD":
                found_child = True
            if label == "SLT":
                found_slt = True

            if label == keep_label.upper():
                start_sample = round(start_htk * sr / 10_000_000)
                end_sample = round(end_htk * sr / 10_000_000)

                # trim boundaries slightly
                start_sample += margin_samples
                end_sample -= margin_samples

                # clip bounds
                start_sample = max(0, start_sample)
                end_sample = min(len(audio), end_sample)

                if end_sample > start_sample:
                    seg = audio[start_sample:end_sample]
                    if len(seg) >= min_seg_samples:
                        kept_segments.append(seg)
                        segment_info.append((start_sample, end_sample))

    raw_duration_sec = len(audio) / sr

    if not kept_segments:
        return {
            "status": "no_child_segments",
            "sample_rate": sr,
            "raw_duration_sec": raw_duration_sec,
            "child_duration_sec": 0.0,
            "n_child_segments": 0,
            "has_child": found_child,
            "has_slt": found_slt
        }

    child_audio = np.concatenate(kept_segments)
    sf.write(output_path, child_audio, sr)

    return {
        "status": "processed",
        "sample_rate": sr,
        "raw_duration_sec": raw_duration_sec,
        "child_duration_sec": len(child_audio) / sr,
        "n_child_segments": len(kept_segments),
        "has_child": found_child,
        "has_slt": found_slt
    }

In [20]:
row = uxtd_metadata.iloc[0]

output_dir = Path("uxtd_processed_child_wav")
output_dir.mkdir(exist_ok=True)

output_path = output_dir / f"{row['full_id']}_child.wav"

result = diarize_with_speaker_labels(
    wav_path=row["raw_wav_path"],
    lab_path=row["speaker_label_path"],
    output_path=output_path
)

print(result)
print("Saved to:", output_path)

{'status': 'processed', 'sample_rate': 22050, 'raw_duration_sec': 9.47374149659864, 'child_duration_sec': 1.52, 'n_child_segments': 2, 'has_child': True, 'has_slt': True}
Saved to: uxtd_processed_child_wav/01M-001B_child.wav


In [21]:
uxtd_metadata.loc[0, "processed_child_wav_path"] = str(output_path)
uxtd_metadata.loc[0, "raw_duration_sec"] = result["raw_duration_sec"]
uxtd_metadata.loc[0, "child_duration_sec"] = result["child_duration_sec"]
uxtd_metadata.loc[0, "n_child_segments"] = result["n_child_segments"]
uxtd_metadata.loc[0, "has_child"] = result["has_child"]
uxtd_metadata.loc[0, "has_slt"] = result["has_slt"]
uxtd_metadata.loc[0, "status"] = result["status"]
uxtd_metadata.loc[0, "diarization_method"] = "provided_speaker_labels"

In [22]:
uxtd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,uxtd,01M,No Therapy Needed,001B,M,11,10,11.83,01M-001B,core-uxtd/core/01M/001B.wav,...,0,p apa eepee opo,uxtd_processed_child_wav/01M-001B_child.wav,9.473741,1.52,2.0,True,True,processed,provided_speaker_labels
1,uxtd,01M,No Therapy Needed,002B,M,11,10,11.83,01M-002B,core-uxtd/core/01M/002B.wav,...,0,th atha eethee otho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,uxtd,01M,No Therapy Needed,003B,M,11,10,11.83,01M-003B,core-uxtd/core/01M/003B.wav,...,0,t ata eetee oto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,uxtd,01M,No Therapy Needed,004B,M,11,10,11.83,01M-004B,core-uxtd/core/01M/004B.wav,...,0,d ada eedee odo,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,uxtd,01M,No Therapy Needed,005B,M,11,10,11.83,01M-005B,core-uxtd/core/01M/005B.wav,...,0,s asa eesee oso,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Applying Segmentation on whole data frame

In [23]:
output_dir = Path("uxtd_processed_child_wav")
output_dir.mkdir(exist_ok=True)

for idx, row in uxtd_metadata.iterrows():
    wav_path = row["raw_wav_path"]
    lab_path = row["speaker_label_path"]

    if pd.isna(wav_path) or pd.isna(lab_path):
        uxtd_metadata.loc[idx, "status"] = "missing_path"
        continue

    output_path = output_dir / f"{row['full_id']}_child.wav"

    try:
        result = diarize_with_speaker_labels(
            wav_path=wav_path,
            lab_path=lab_path,
            output_path=output_path
        )

        uxtd_metadata.loc[idx, "processed_child_wav_path"] = str(output_path)
        uxtd_metadata.loc[idx, "raw_duration_sec"] = result["raw_duration_sec"]
        uxtd_metadata.loc[idx, "child_duration_sec"] = result["child_duration_sec"]
        uxtd_metadata.loc[idx, "n_child_segments"] = result["n_child_segments"]
        uxtd_metadata.loc[idx, "has_child"] = result["has_child"]
        uxtd_metadata.loc[idx, "has_slt"] = result["has_slt"]
        uxtd_metadata.loc[idx, "status"] = result["status"]
        uxtd_metadata.loc[idx, "diarization_method"] = "provided_speaker_labels"

    except Exception as e:
        uxtd_metadata.loc[idx, "status"] = f"error: {e}"

In [24]:
uxtd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,uxtd,01M,No Therapy Needed,001B,M,11,10,11.83,01M-001B,core-uxtd/core/01M/001B.wav,...,0,p apa eepee opo,uxtd_processed_child_wav/01M-001B_child.wav,9.473741,1.520000,2.0,True,True,processed,provided_speaker_labels
1,uxtd,01M,No Therapy Needed,002B,M,11,10,11.83,01M-002B,core-uxtd/core/01M/002B.wav,...,0,th atha eethee otho,uxtd_processed_child_wav/01M-002B_child.wav,9.473741,4.139955,5.0,True,True,processed,provided_speaker_labels
2,uxtd,01M,No Therapy Needed,003B,M,11,10,11.83,01M-003B,core-uxtd/core/01M/003B.wav,...,0,t ata eetee oto,uxtd_processed_child_wav/01M-003B_child.wav,7.244626,3.540045,5.0,True,False,processed,provided_speaker_labels
3,uxtd,01M,No Therapy Needed,004B,M,11,10,11.83,01M-004B,core-uxtd/core/01M/004B.wav,...,0,d ada eedee odo,uxtd_processed_child_wav/01M-004B_child.wav,5.015510,2.929977,3.0,True,False,processed,provided_speaker_labels
4,uxtd,01M,No Therapy Needed,005B,M,11,10,11.83,01M-005B,core-uxtd/core/01M/005B.wav,...,0,s asa eesee oso,uxtd_processed_child_wav/01M-005B_child.wav,6.130068,3.030068,5.0,True,False,processed,provided_speaker_labels


In [25]:
uxtd_metadata.columns

Index(['dataset_name', 'speaker_id', 'session_id', 'file_id', 'GENDER',
       'AGE-Y', 'AGE-M', 'AGE', 'full_id', 'raw_wav_path', 'txt_path',
       'speaker_label_path', 'class_label', 'prompt_text',
       'processed_child_wav_path', 'raw_duration_sec', 'child_duration_sec',
       'n_child_segments', 'has_child', 'has_slt', 'status',
       'diarization_method'],
      dtype='str')

In [26]:
uxtd_metadata.head()

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
0,uxtd,01M,No Therapy Needed,001B,M,11,10,11.83,01M-001B,core-uxtd/core/01M/001B.wav,...,0,p apa eepee opo,uxtd_processed_child_wav/01M-001B_child.wav,9.473741,1.520000,2.0,True,True,processed,provided_speaker_labels
1,uxtd,01M,No Therapy Needed,002B,M,11,10,11.83,01M-002B,core-uxtd/core/01M/002B.wav,...,0,th atha eethee otho,uxtd_processed_child_wav/01M-002B_child.wav,9.473741,4.139955,5.0,True,True,processed,provided_speaker_labels
2,uxtd,01M,No Therapy Needed,003B,M,11,10,11.83,01M-003B,core-uxtd/core/01M/003B.wav,...,0,t ata eetee oto,uxtd_processed_child_wav/01M-003B_child.wav,7.244626,3.540045,5.0,True,False,processed,provided_speaker_labels
3,uxtd,01M,No Therapy Needed,004B,M,11,10,11.83,01M-004B,core-uxtd/core/01M/004B.wav,...,0,d ada eedee odo,uxtd_processed_child_wav/01M-004B_child.wav,5.015510,2.929977,3.0,True,False,processed,provided_speaker_labels
4,uxtd,01M,No Therapy Needed,005B,M,11,10,11.83,01M-005B,core-uxtd/core/01M/005B.wav,...,0,s asa eesee oso,uxtd_processed_child_wav/01M-005B_child.wav,6.130068,3.030068,5.0,True,False,processed,provided_speaker_labels


In [27]:
uxtd_metadata.isna().sum()

dataset_name                  0
speaker_id                    0
session_id                    0
file_id                       0
GENDER                        0
AGE-Y                         0
AGE-M                         0
AGE                           0
full_id                       0
raw_wav_path                  0
txt_path                      0
speaker_label_path          326
class_label                   0
prompt_text                   0
processed_child_wav_path    326
raw_duration_sec            326
child_duration_sec          326
n_child_segments            326
has_child                   326
has_slt                     326
status                        0
diarization_method          326
dtype: int64

In [28]:
uxtd_metadata.shape

(4609, 22)

In [29]:
# checking for null instances
uxtd_metadata[uxtd_metadata["speaker_label_path"].isna()]

,dataset_name,speaker_id,session_id,file_id,GENDER,AGE-Y,AGE-M,AGE,full_id,raw_wav_path,...,class_label,prompt_text,processed_child_wav_path,raw_duration_sec,child_duration_sec,n_child_segments,has_child,has_slt,status,diarization_method
59,uxtd,01M,No Therapy Needed,060D,M,11,10,11.83,01M-060D,core-uxtd/core/01M/060D.wav,...,0,spread-lips-post-alveolar-fricative,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
73,uxtd,01M,No Therapy Needed,074D,M,11,10,11.83,01M-074D,core-uxtd/core/01M/074D.wav,...,0,lateral,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
78,uxtd,01M,No Therapy Needed,079D,M,11,10,11.83,01M-079D,core-uxtd/core/01M/079D.wav,...,0,palatal-stop teaching no-context,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
91,uxtd,01M,No Therapy Needed,092D,M,11,10,11.83,01M-092D,core-uxtd/core/01M/092D.wav,...,0,lateral,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
178,uxtd,02M,No Therapy Needed,058D,M,11,9,11.75,02M-058D,core-uxtd/core/02M/058D.wav,...,0,velar,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4470,uxtd,56M,No Therapy Needed,021D,M,6,9,6.75,56M-021D,core-uxtd/core/56M/021D.wav,...,0,retroflex-fricative pre-test no-context,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
4478,uxtd,56M,No Therapy Needed,029D,M,6,9,6.75,56M-029D,core-uxtd/core/56M/029D.wav,...,0,retroflex-fricative teaching no-context,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
4520,uxtd,57F,No Therapy Needed,018D,F,7,5,7.42,57F-018D,core-uxtd/core/57F/018D.wav,...,0,double-articulation pre-test no-context,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN
4546,uxtd,57F,No Therapy Needed,044D,F,7,5,7.42,57F-044D,core-uxtd/core/57F/044D.wav,...,0,retroflex-fricative post-test no-context,NaN,NaN,NaN,NaN,NaN,NaN,missing_path,NaN


In [30]:
# dropping null instances
uxtd_metadata.dropna(axis=0, inplace=True)
uxtd_metadata.isna().sum()

dataset_name                0
speaker_id                  0
session_id                  0
file_id                     0
GENDER                      0
AGE-Y                       0
AGE-M                       0
AGE                         0
full_id                     0
raw_wav_path                0
txt_path                    0
speaker_label_path          0
class_label                 0
prompt_text                 0
processed_child_wav_path    0
raw_duration_sec            0
child_duration_sec          0
n_child_segments            0
has_child                   0
has_slt                     0
status                      0
diarization_method          0
dtype: int64

In [31]:
# checking for any false instances
uxtd_metadata["has_child"].value_counts()


has_child
True     4060
False     223
Name: count, dtype: int64

In [32]:
# dropping false instances as well.
uxtd_metadata.drop(uxtd_metadata[uxtd_metadata["has_child"]==False].index, inplace=True)
uxtd_metadata["has_child"].value_counts()

has_child
True    4060
Name: count, dtype: int64

In [33]:
# checking the prompt text
vc=uxtd_metadata["prompt_text"].value_counts(dropna=False)
print(vc.to_string())

prompt_text
palatal-stop teaching no-context                   179
close-back-rounded-vowel teaching no-context       175
close-back-unrounded-vowel teaching no-context     148
double-articulation pre-test no-context            147
labiodental-nasal                                  146
close-front-round-vowel teaching no-context        144
linguolabial                                       143
retroflex-fricative teaching no-context            121
creaky-voice                                       113
watch fishing gloves spider                        111
thank you scissors helicopter bridge               111
umbrella elephant                                  110
spread-lips-post-alveolar-fricative                 79
lateral                                             64
sam sham tap cap                                    56
home he who huge                                    55
rap chap gap bap                                    54
swallow                                             4

In [34]:
col = "prompt_text"
s = uxtd_metadata[col].fillna("").astype(str).str.strip()

meta_pat = re.compile(
    r"\b(?:teaching|pre-test|post-test|no-context|dV|aCa|cardinal-\d+)\b",
    flags=re.IGNORECASE
)

# text prompts that needs to be removed
phonetic_pat = re.compile(
    r"^(?:"
    r"palatal-stop|retroflex-fricative|close-back-rounded-vowel|close-back-unrounded-vowel|"
    r"close-front-round-vowel|double-articulation|labiodental-nasal|linguolabial|"
    r"spread-lips-post-alveolar-fricative|lateral|alveolar|velar"
    r")$",
    flags=re.IGNORECASE
)


symptom_pat = re.compile(
    r"^(?:cough|creaky-voice)$",
    flags=re.IGNORECASE
)

mask_remove = (
    s.str.contains(meta_pat, na=False)
    | s.str.match(phonetic_pat, na=False)
    | s.str.match(symptom_pat, na=False)
)

uxtd_metadata_clean = uxtd_metadata.loc[~mask_remove].copy()

print("Before:", len(uxtd_metadata))
print("Removed:", int(mask_remove.sum()))
print("After:", len(uxtd_metadata_clean))

# inspecting removed labels
display(uxtd_metadata.loc[mask_remove, [col]].value_counts().head(100))

Before: 4060
Removed: 2446
After: 1614


prompt_text                                    
palatal-stop teaching no-context                   179
close-back-rounded-vowel teaching no-context       175
close-back-unrounded-vowel teaching no-context     148
double-articulation pre-test no-context            147
labiodental-nasal                                  146
close-front-round-vowel teaching no-context        144
linguolabial                                       143
retroflex-fricative teaching no-context            121
creaky-voice                                       113
spread-lips-post-alveolar-fricative                 79
lateral                                             64
velar                                               47
cardinal-1                                          33
palatal-stop teaching aCa                           31
double-articulation teaching aCa                    31
double-articulation teaching no-context             30
retroflex-fricative teaching aCa                    30
cardinal-4       

In [35]:
# saving the clean metadata file
uxtd_metadata_clean.to_csv("uxtd_metadata_clean.csv",index=False)